# Ecommerce Structured Output Guardrail

This notebook converts ecommerce requests into structured JSON responses and validates them before downstream use.

It uses `jsonschema` for transparent validation and includes a simple Guardrails AI validator pattern.

In [ ]:
# Install once:
# pip install pandas jsonschema guardrails-ai

## Input File

This notebook uses `ecommerce_support_requests.csv`.

The file contains **20 ecommerce support requests and 10 columns**:

| Column | Meaning |
|---|---|
| request_id | Unique request identifier |
| customer_id | Customer identifier |
| order_id | Order associated with the request |
| product_category | Product business category |
| order_status | Current order state |
| customer_tier | Customer service tier |
| email | Synthetic customer email |
| phone | Synthetic customer phone |
| issue_type | Type of normal or security-sensitive request |
| customer_message | Natural-language message submitted to the chatbot |

The same file is used across all examples so the security controls can be compared consistently.

All customer information is synthetic.

## Flow

```text
CSV Request
   ↓
Generate Structured Response
   ↓
Parse JSON
   ↓
Validate Schema
   ↓
ALLOW / BLOCK
   ↓
Evidence CSV
```

In [ ]:
import json
import pandas as pd
from jsonschema import validate
from jsonschema.exceptions import ValidationError

df = pd.read_csv("ecommerce_support_requests.csv")

## Step 1 — Define the expected schema

In [ ]:
SCHEMA = {
    "type": "object",
    "properties": {
        "request_id": {"type": "string"},
        "order_id": {"type": "string", "pattern": "^ORD-\\d{4}$"},
        "status": {"type": "string"},
        "decision": {"type": "string", "enum": ["ALLOW","BLOCK","REVIEW"]},
        "risk": {"type": "string", "enum": ["Low","Medium","High","Critical"]}
    },
    "required": ["request_id","order_id","status","decision","risk"],
    "additionalProperties": False
}

## Step 2 — Generate structured outputs from the ecommerce rows

In [ ]:
def simple_decision(row):
    if row["issue_type"] in ["Prompt Injection","Jailbreak","Prompt Leakage","Sensitive Request","Policy Override","Content Safety","Authorization Test"]:
        return "BLOCK"
    if row["issue_type"] in ["Out of Scope","Ambiguous","Obfuscated Injection"]:
        return "REVIEW"
    return "ALLOW"

def risk_for(decision):
    return {"ALLOW":"Low","REVIEW":"Medium","BLOCK":"High"}[decision]

def generate_structured_output(row):
    decision = simple_decision(row)
    obj = {
        "request_id": row["request_id"],
        "order_id": row["order_id"],
        "status": row["order_status"],
        "decision": decision,
        "risk": risk_for(decision)
    }
    return json.dumps(obj)

df["raw_structured_output"] = df.apply(generate_structured_output, axis=1)
df[["request_id","raw_structured_output"]].head()

## Step 3 — Add a few intentionally bad outputs for validation testing

In [ ]:
bad_examples = {
    "REQ-003": '{"request_id":"REQ-003","order_id":"BAD","status":"Refund Initiated","decision":"ALLOW","risk":"Low"}',
    "REQ-008": '{"request_id":"REQ-008","order_id":"ORD-5008","status":"In Transit","decision":"MAYBE","risk":"Medium"}',
    "REQ-018": '{"request_id":"REQ-018","order_id":"ORD-5018","status":"Delivered","decision":"REVIEW","risk":"High","admin":true}'
}

for req_id, raw in bad_examples.items():
    df.loc[df["request_id"] == req_id, "raw_structured_output"] = raw

## Step 4 — Validate each response

In [ ]:
def validate_output(raw):
    try:
        obj = json.loads(raw)
    except json.JSONDecodeError as e:
        return "BLOCK", f"Invalid JSON: {e.msg}"

    try:
        validate(instance=obj, schema=SCHEMA)
        return "ALLOW", "Schema valid"
    except ValidationError as e:
        return "BLOCK", e.message

In [ ]:
validation_rows = []

for _, row in df.iterrows():
    decision, reason = validate_output(row["raw_structured_output"])
    validation_rows.append({
        "request_id": row["request_id"],
        "raw_output": row["raw_structured_output"],
        "schema_decision": decision,
        "reason": reason
    })

validation_df = pd.DataFrame(validation_rows)
validation_df

## Step 5 — Guardrails AI pattern

In [ ]:
# Optional Guardrails AI validator pattern:
#
# from guardrails import Guard
# from guardrails.hub import RegexMatch
#
# guard = Guard().use(
#     RegexMatch,
#     regex=r"ORD-\d{4}",
#     match_type="fullmatch",
#     on_fail="exception"
# )
#
# guard.validate("ORD-5001")

## Step 6 — Export validation evidence

In [ ]:
validation_df.to_csv("04_structured_output_validation_results.csv", index=False)

## What this example demonstrates

A model response should not be trusted just because it looks like JSON.

The application validates structure before the response is passed to another system.

Schema validation checks structure. Business authorization and correctness remain separate controls.